# §41: **K5f / K5dfi / L0f / L0dfi** trained on MSAD-full (whole videos), head kernel 3 **and** 9, s2024 — a clean CCTV-trained version

Rule: `core/docs/final/KIM_RUN.md` **§41**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. PRELIMINARY (n = 1); **nothing here picks a kernel or adopts a row** (R6), and the report never prints a best-of.

**Videos:** the user's shortcut `Thesis-Final/data/MSAD Videos blurred` (official blurred release): `MSAD_anomaly/`, `MSAD_normal_training/` (never `VALU_MSAD_subset_2026`). **Everything is MSAD-scaled (Amendment 1): the motion statistics are fitted on MSAD train, and no Nexar-trained row is read.** **Needs on Drive before this runs:** `Thesis/data/MSAD/{labels_train,frame_labels_test,meta,defs}.json`, `Thesis/cache/clip/MSAD_ncc` (train and test videos), **the test videos' V2-S rows `…/video/…/MSAD-full_s8_squash` (§18, `k5_msad.ipynb` step 9)**, for TAD `Thesis-V2/cache/video/…/TAD_s8_squash` + `Thesis/cache/clip/TAD_ncc`, the Nexar V2-S manifest (geometry check) and the saved `config.yaml` of the Nexar rows K5f / K5dfi / L0dfi (recipe check only).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, helpers, preflight (controls' J10, inputs exist) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 2 min |
| 5 | copy the 480 train videos -> VM; **train census + gate M2 (+ label/folder agreement, length ruler)** | no | 10-20 min |
| 6 | decode -> V2-S s8 in chunks (frames deleted per chunk) -> `…/MSAD-full_s8_squash` | **yes** | 30-60 min |
| 7 | **fit** the A2 statistics on the 480 MSAD train videos, bake train / test / TAD with them -> `…/A2_S/MSAD-full_{train,test}_s8_msad`, `TAD_s8_msad` | no | 3 min |
| 8 | KNN filler cache (DVS rows) | no | 1 min |
| 9 | plan + preflight numbers (median rows, k, steps) | no | < 1 min |
| 10 | train the rows (500 steps each, J10 + config diff vs the Nexar parent) | **yes** | **8 rows — measure s/step on the first row** |
| 11 | MSAD test reads of the 8 rows | yes | ~3 min per row |
| 12 | TAD reads (optional, `DO_TAD`) | yes | ~3 min per row |
| 13 | the table -> **paste `REPORTS/k5msad_train_report.md` to Claude** | no | 1 min |

`ROWS` (below) can be cut to a subset to train in several sessions; every step skips what is already on Drive. **MSAD rows are up to 750 steps long (Nexar's windows are 20), so s/step is higher than K5's: read it off the first log lines.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data/MSAD, cache/clip/{MSAD_ncc,TAD_ncc}
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: TAD V2-S rows, Nexar V2-S manifest
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, the Nexar rows' configs, the MSAD shortcut, every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 41. K5f, K5dfi, L0f and L0dfi' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §41 -- re-upload kat-vad/'
assert (REPO / 'core/tools/msad_train.py').is_file(), 'no msad_train.py -- re-upload kat-vad/'
assert 'SPLIT_TRAIN' in (REPO / 'core/tools/msad_zs.py').read_text(), 'msad_zs has no --split train -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
from core.tools import msad_train as mt  # noqa: E402
from core.tools import msad_zs  # noqa: E402

# ---- what to run --------------------------------------------------------------------------------------
ROWS     = mt.all_rows()          # K5fM3 K5dfiM3 L0fM3 L0dfiM3 K5fM9 ... ; cut this list to train in several sessions
DO_TAD   = True                   # §41 R4: TAD reads (macro over 60) for the rows
CHUNK    = 60                     # videos decoded -> V2-S -> deleted per chunk (VM disk)

K5       = constants.KIM_MAIN_ARM                   # "K5": names run / eval sub-folders only
SEED     = constants.KIM_SEED                       # 2024
ENC      = constants.V5_ENCODER                     # V2-S
OFF      = constants.V2_OFF
DSET     = constants.MSAD_FULL_DATASET              # 'MSAD-full'
STRIDE   = constants.FRAME_STRIDE                   # 8
VSTEP    = constants.VIDEOMAE_CLIP_FRAME_STEP       # 3 (Nexar geometry)
FPS      = constants.MSAD_ZS_ASSUMED_FPS            # 30
TAD_NAME = constants.TAD_DATASET

VIDEOS   = Path(DRIVE_F) / 'data' / constants.MSAD_ZS_VIDEOS_DIRNAME       # the user's shortcut (read-only)
DATA     = Path(DRIVE) / 'data' / 'MSAD'                                    # labels + defs, as full_gate_a
CLIP     = Path(DRIVE) / 'cache' / 'clip' / 'MSAD_ncc'                      # CLIP of train and test videos
TAD_V2S  = Path(DRIVE_V2) / 'cache' / 'video' / ENC / f'{TAD_NAME}_s{STRIDE}_squash'   # TAD's V2-S rows, row-aligned with TAD_ncc
TAD      = constants.DATA_ROOT / TAD_NAME
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'
NEXAR_V2S = Path(DRIVE_V2) / 'cache' / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
V2S      = constants.CACHE_ROOT / 'video' / ENC / f'{DSET}_s{STRIDE}_squash'                      # §18's dir: test + train ids
A2_DIR     = constants.CACHE_ROOT / 'v2' / 'A2_S'
BAKE_TRAIN = A2_DIR / constants.MSAD_TRAIN_BAKE_DIRNAME     # the statistics FITTED on MSAD train + the 480 train videos (Amendment 1)
BAKE_TEST  = A2_DIR / constants.MSAD_TEST_BAKE_DIRNAME      # the 240 test videos, MSAD's statistics
BAKE_TAD   = A2_DIR / constants.MSAD_TAD_BAKE_DIRNAME       # TAD, MSAD's statistics
KNN      = constants.CACHE_ROOT / 'knn' / 'MSAD_ncc' / constants.KNN_CACHE_FILENAME
KNN_V1   = Path(DRIVE) / 'cache' / 'knn' / 'MSAD_ncc' / constants.KNN_CACHE_FILENAME                # v1's, reused if it checks out

ROOT     = constants.OUTPUT_ROOT / constants.KIM_MSAD_TRAIN_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
EVAL, EVAL_TAD = ROOT / 'eval', ROOT / 'eval_tad'
CENSUS   = REPORTS / constants.MSAD_TRAIN_CENSUS_FILENAME
NEXAR_RUN = {                                       # the parents of the config guard (config.yaml only: no checkpoint is read)
    'K5f': constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME / constants.KIM_REFERENCE_ARM / K5 / f's{SEED}',
    'K5dfi': constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / constants.KIM_DVS_IGNORE_ROW / K5 / f's{SEED}',
    'L0dfi': constants.OUTPUT_ROOT / constants.KIM_LAGOVAD_NEXAR_OUTPUT_DIRNAME / constants.KIM_LAGOVAD_NEXAR_ROW / K5 / f's{SEED}',
}
PARENT_RUN = {arm: NEXAR_RUN[mt.PARENT[arm]] for arm in mt.ARM_SETS}   # L0f's parent is K5f

LOCAL    = Path('/content/k5_msad_train')
L_VIDEOS, FRAMES = LOCAL / 'videos', LOCAL / 'frames'                   # VM disk, never Drive
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

for name, path in (('MSAD shortcut', VIDEOS), ('labels_train', DATA / constants.LABELS_TRAIN_FILENAME),
                   ('frame_labels_test', DATA / constants.FRAME_LABELS_TEST_FILENAME), ('MSAD defs', DATA / constants.DEFS_FILENAME),
                   ('MSAD meta', DATA / constants.META_FILENAME), ('MSAD_ncc', CLIP),
                   ('Nexar V2-S manifest', NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME),
                   *[(f'{c} config', NEXAR_RUN[c] / 'config.yaml') for c in NEXAR_RUN]):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:24s} {path}')
if DO_TAD:
    for name, path in (('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('TAD V2-S', TAD_V2S), ('TAD_ncc', TAD_NCC)):
        assert path.exists(), f'missing {name}: {path}'

def on_drive_early(folder):
    return {e.name[:-4] for e in os.scandir(folder) if e.name.endswith('.npy')} if folder.is_dir() else set()


LABELS_TRAIN = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
TRAIN_IDS = sorted(LABELS_TRAIN)
TEST_IDS = sorted(json.loads((DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
N_POS = sum(LABELS_TRAIN.values())
assert (len(TRAIN_IDS), N_POS) == (constants.MSAD_TRAIN_VIDEOS, constants.MSAD_TRAIN_ABNORMAL), (len(TRAIN_IDS), N_POS)
assert len(TEST_IDS) == constants.MSAD_ZS_TEST_VIDEOS, len(TEST_IDS)
assert not set(TRAIN_IDS) & set(TEST_IDS), 'a video is in both train and test'
assert set(TEST_IDS) <= on_drive_early(V2S), 'the test videos have no V2-S rows: run k5_msad.ipynb steps 5-9 first (§18)'
print(f'{len(TRAIN_IDS)} train videos ({N_POS} abnormal), {len(TEST_IDS)} test; rows {ROWS}; outputs -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd pyyaml
python -c "import av, einops, faiss, sklearn, yaml; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6, 10, 11, 12 need one'
df -h /content | tail -1

In [ ]:
import numpy as np
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.knn_cache import load_knn_cache       # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF
DEFAULTS = _flat(Config().to_dict())


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    '''Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename).'''
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False)['global_step'])
    except Exception:  # noqa: BLE001 -- a truncated Drive upload must read as "not final"
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


# Preflight on small files only (C39): the parents' configs must be the recipes this section copies.
for name, run_dir in NEXAR_RUN.items():
    cfg = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    assert cfg['kim.enabled'] is (name != 'L0dfi'), f'{name}: kim.enabled {cfg["kim.enabled"]}'
    assert cfg['dvs.theta'] == (1.0 if name == 'K5f' else 0.7), f'{name}: dvs.theta {cfg["dvs.theta"]}'
    print(f'{name}: config.yaml is the {name} recipe')
L_DATA = LOCAL / 'stage' / 'data'                    # json only -- never a video folder
L_DATA.mkdir(parents=True, exist_ok=True)
for f in DATA.iterdir():
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_DATA / f.name)
TRAIN_FILE = write_ids(LOCAL / 'ids' / 'train.txt', TRAIN_IDS)
print('helpers + preflight OK')

## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_msad_train.py',
     'core/tests/test_msad_zs.py', 'core/tests/test_k5_msad_report.py'])

## 5. Train census + gate M2 (`KIM_RUN.md` §41 R7) — stops here on any failure

Copies the 480 train videos from the shortcut's **official folders** (`MSAD_anomaly/`, `MSAD_normal_training/`; never `VALU_MSAD_subset_2026/`) to VM disk, then gates: each `labels_train.json` id found **exactly once** (a normal train video may be an `.avi`), 120 label-1 ids, **each label agrees with the folder**, `ceil(decoded / 8) == rows(MSAD_ncc)`. Prints rows by label, the **train length ruler** (P(abnormal shorter than normal)) and the fps spread.

In [ ]:
NEED_VIDEOS = not (CENSUS.exists() and msad_zs.census_current(json.loads(CENSUS.read_text()), constants.MSAD_TRAIN_SOURCE_DIRS)
                   and set(TRAIN_IDS) <= on_drive(V2S) and (BAKE_TRAIN / constants.V2_INPUT_MANIFEST_FILENAME).exists())
if NEED_VIDEOS:
    want, t0, n = set(TRAIN_IDS), time.time(), 0
    for d in constants.MSAD_TRAIN_SOURCE_DIRS:
        for suffix in constants.MSAD_VIDEO_SUFFIXES:
            for p in (VIDEOS / d).rglob(f'*{suffix}'):
                if p.stem in want:
                    dst = L_VIDEOS / p.relative_to(VIDEOS)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        tmp = dst.with_name(dst.name + '.partial')
                        shutil.copy2(p, tmp)
                        os.replace(tmp, dst)
                    n += 1
    print(f'{n} files on VM disk in {(time.time() - t0) / 60:.1f} min (the census checks each id is there exactly once)')
    if not (CENSUS.exists() and msad_zs.census_current(json.loads(CENSUS.read_text()), constants.MSAD_TRAIN_SOURCE_DIRS)):
        local = LOCAL / constants.MSAD_TRAIN_CENSUS_FILENAME
        try:
            run([sys.executable, '-m', 'core.tools.msad_zs', 'census', '--split', 'train', '--videos-root', L_VIDEOS,
                 '--labels', DATA / constants.LABELS_TRAIN_FILENAME, '--clip-dir', CLIP, '--stride', STRIDE, '--out', local],
                log=LOCAL / 'census.log')
        finally:
            if local.exists():
                publish(local, CENSUS)
else:
    print('census, V2-S and the bake are on Drive -- nothing copied')
doc = json.loads(CENSUS.read_text())
print(json.dumps({k: v for k, v in doc['summary'].items() if k != 'failures'}, indent=1))
assert doc['summary']['pass'], f"MSAD train census FAILED (M2): {doc['summary']['failures']} -- paste this to Claude"
CENSUS_ROWS = {v: e['rows'] for v, e in doc['videos'].items()}

## 6. Decode -> VideoMAE V2-S at s8, frame step 3, 30 fps (Nexar's geometry), **in chunks**

`CHUNK` videos are decoded to JPEG, run through V2-S, published to `…/video/…/MSAD-full_s8_squash` (the directory §18 wrote the test ids to) and the frames are deleted, so the 480 train videos never sit on VM disk at once. Resumable: a chunk whose ids are all on Drive is skipped.

In [ ]:
todo = [v for v in TRAIN_IDS if v not in on_drive(V2S)]
chunks = [todo[i:i + CHUNK] for i in range(0, len(todo), CHUNK)]
print(f'{len(TRAIN_IDS) - len(todo)} of {len(TRAIN_IDS)} train ids already have V2-S; {len(chunks)} chunk(s) to do')
if chunks:
    assert torch.cuda.is_available(), 'no GPU'
for i, ids in enumerate(chunks):
    t0 = time.time()
    chunk_file = write_ids(LOCAL / 'ids' / f'chunk_{i}.txt', ids)
    shutil.rmtree(FRAMES, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.msad_zs', 'decode', '--videos-root', L_VIDEOS, '--census', CENSUS,
         '--out-dir', FRAMES, '--ids-file', chunk_file], log=LOCAL / 'decode.log')
    local = LOCAL / 'v2s' / f'chunk_{i}'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.extract_video_features', '--frames-dir', FRAMES, '--ids-file', chunk_file,
         '--dataset', DSET, '--encoder', ENC, '--stride', STRIDE, '--clip-frame-step', VSTEP,
         '--assumed-fps', FPS, '--batch-size', 32, '--device', 'auto', '--output-dir', local], log=LOCAL / 'v2s.log')
    publish(local, V2S)
    shutil.rmtree(FRAMES, ignore_errors=True)
    shutil.rmtree(local, ignore_errors=True)
    print(f'chunk {i + 1}/{len(chunks)}: {len(ids)} videos in {(time.time() - t0) / 60:.1f} min')
if chunks:
    flush_drive()
mine = json.loads((V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
ref = json.loads((NEXAR_V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
diff = sorted(k for k in set(mine) | set(ref) if mine.get(k) != ref.get(k))
assert set(diff) <= {'stride'} and mine['stride'] == STRIDE, f"MSAD V2-S manifest differs from Nexar's on {diff} (§41 R7)"
L_CLIP_TRAIN = stage(CLIP, LOCAL / 'stage' / 'clip_train', TRAIN_IDS)
L_V2S_TRAIN = stage(V2S, LOCAL / 'stage' / 'v2s_train', TRAIN_IDS)
bad = [v for v in TRAIN_IDS if not rows(L_V2S_TRAIN, v) == rows(L_CLIP_TRAIN, v) == CENSUS_ROWS[v]]
assert not bad, f'V2-S rows != MSAD_ncc rows != census rows on {bad[:5]} (M3, C13)'
print(f"V2-S OK: manifest == Nexar's except {diff}; rows == MSAD_ncc rows == census rows on {len(TRAIN_IDS)} videos")

## 7. **Fit** the A2 statistics (no CRN, V2-S) on the 480 MSAD train videos, bake train, test and TAD with them (CPU)

Amendment 1: the point is a clean CCTV-trained version, so the motion statistics (`m_u`, `sigma_u`, `c`) come from MSAD train, not Nexar. The manifest's `train_ids_sha1` must equal the sha1 of `labels_train.json`'s ids; the test and TAD bakes load the fitted statistics and refit nothing.

In [ ]:
from core.data.v2_splits import lines_sha1  # noqa: E402

man_path = BAKE_TRAIN / constants.V2_INPUT_MANIFEST_FILENAME
if not (man_path.exists() and json.loads(man_path.read_text())['baked_ids'] == len(TRAIN_IDS)):
    local = LOCAL / 'bake_train'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit-ids', '--train-ids-file', TRAIN_FILE, '--ids-file', TRAIN_FILE,
         '--clip-dir', L_CLIP_TRAIN, '--video-dir', L_V2S_TRAIN, '--stride', 1, '--crn', OFF, '--motion', ENC,
         '--out-dir', local], log=LOCAL / 'bake_train.log')
    publish(local, BAKE_TRAIN)
    flush_drive()
m = manifest(BAKE_TRAIN)
SHA = lines_sha1(TRAIN_IDS)
assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA) and m['baked_ids'] == len(TRAIN_IDS), m
L_BAKE_TRAIN = stage(BAKE_TRAIN, LOCAL / 'stage' / 'bake_train', TRAIN_IDS)
bad = [v for v in TRAIN_IDS if rows(L_BAKE_TRAIN, v) != rows(L_CLIP_TRAIN, v)]
assert not bad, f'bake rows != CLIP rows on {bad[:5]} (M3)'
STATS = stage(BAKE_TRAIN, LOCAL / 'stage' / 'msad_stats', [])          # manifest + fitted statistics only


def apply_bake(name, clip_src, v2s_src, ids, dst):
    man = dst / constants.V2_INPUT_MANIFEST_FILENAME
    if not (man.exists() and json.loads(man.read_text())['baked_ids'] == len(ids)):
        l_clip = stage(clip_src, LOCAL / 'stage' / f'{name}_clip', ids)
        l_v2s = stage(v2s_src, LOCAL / 'stage' / f'{name}_v2s', ids)
        bad = [v for v in ids if rows(l_v2s, v) != rows(l_clip, v)]
        assert not bad, f'{name}: V2-S rows != CLIP rows on {bad[:5]} (M3, C13)'
        local = LOCAL / f'bake_{name}'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', STATS, '--clip-dir', l_clip,
             '--video-dir', l_v2s, '--ids-file', write_ids(LOCAL / 'ids' / f'{name}.txt', ids), '--stride', 1,
             '--out-dir', local], log=LOCAL / f'bake_{name}.log')
        publish(local, dst)
        flush_drive()
    mm = manifest(dst)
    assert (mm['crn'], mm['motion'], mm['train_ids_sha1']) == (OFF, ENC, SHA) and mm['baked_ids'] == len(ids), mm
    print(f'{name} bake OK: {len(ids)} videos, statistics fitted on MSAD train {SHA[:12]}')


apply_bake('msad_test', CLIP, V2S, TEST_IDS, BAKE_TEST)
if DO_TAD:
    TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
    apply_bake('tad', TAD_NCC, TAD_V2S, TAD_IDS, BAKE_TAD)
print(f'bake OK: A2 (no CRN, {ENC}), statistics fitted on MSAD train {SHA[:12]}')

## 8. KNN filler cache (DVS rows only): for every abnormal train video its nearest normal train videos on the central CLIP frame, no motion key (fixed camera)

v1's `cache/knn/MSAD_ncc/knn_cache.npz` is reused **only if it checks out** against `labels_train.json` (anchors == the 120 abnormal ids, every neighbour a normal train id); otherwise it is rebuilt here.

In [ ]:
ABN = {v for v, y in LABELS_TRAIN.items() if y == 1}
NOR = set(LABELS_TRAIN) - ABN


def knn_ok(path):
    if not path.exists():
        return False
    d = load_knn_cache(path)
    return set(d) == ABN and all(set(nb) <= NOR for nb in d.values())


KNN_USE = next((p for p in (KNN, KNN_V1) if knn_ok(p)), None)
if KNN_USE is None:
    local = LOCAL / 'knn' / constants.KNN_CACHE_FILENAME
    local.parent.mkdir(parents=True, exist_ok=True)
    run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', L_DATA, '--dataset', DSET, '--clip-dir', L_CLIP_TRAIN,
         '--output', local], log=LOCAL / 'knn.log')
    publish(local, KNN)
    KNN_USE = KNN
    assert knn_ok(KNN), 'the rebuilt KNN cache does not match labels_train.json'
L_KNN = stage(KNN_USE, LOCAL / 'stage' / 'knn' / constants.KNN_CACHE_FILENAME)
print(f'KNN: {KNN_USE} ({len(ABN)} anchors)')

## 9. Plan and preflight numbers (printed, not gated beyond §41 R7)

In [ ]:
PLAN = mt.epoch_plan(N_POS)
EPOCHS, STEPS = PLAN['epochs'], PLAN['steps']
med = float(np.median(list(CENSUS_ROWS.values())))
tr = doc['summary']['train']
print(f'plan {PLAN}')
print(f"rows: median {med:.0f} (abnormal {tr['rows_median_abnormal']:.0f}, normal {tr['rows_median_normal']:.0f}), min {tr['rows_min']}, max {tr['rows_max']}")
print(f"MIL top-k = rows // {constants.KIM_MSAD_TRAIN_MIL_TOPK_PCT}: k = {int(med) // constants.KIM_MSAD_TRAIN_MIL_TOPK_PCT} at the median video "
      f"(K5f's pct 5 would be {int(med) // 5}; Nexar windows: 20 rows -> {20 // 5})")
print(f"train length ruler AUC (P(abnormal shorter than normal)): {tr['length_ruler_auc']:.4f}  (> 0.5 = a clip-length shortcut in the supervision, C28)")
print(f"fps spread {doc['summary']['fps_spread']} | off 30 +- 1: {doc['summary']['fps_off_nominal']}")
assert STEPS == 500, f'plan is {STEPS} steps, §41 R2 says 500 (4 steps x 125 epochs)'

## 10. Train the rows (§41 R1 / R2): `core.train` with `msad_train.train_sets`

**Watch the first log lines for s/step** (MSAD videos are up to 750 rows long). After each row: J10 (ckpt step == last metrics step == 500), the **config diff against the Nexar parent** (`msad_train.check_config`: only `data.dataset`, `train.num_epochs`, `loss.mil_topk_pct`, `model.score_head_kernel`; L0f also `kim.*` / `model.motion_dim` / `v2.motion`), last-epoch `mul_mil` > 0.01.

In [ ]:
PARENT_CFG = {arm: _flat(yaml.safe_load((PARENT_RUN[arm] / 'config.yaml').read_text())) for arm in mt.ARM_SETS}


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt'


def check_run(row):
    arm, kernel = mt.parse_row(row)
    run_dir, ckpt = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    diff = mt.check_config(row, PARENT_CFG[arm], got, DEFAULTS, SEED)
    last = [r for r in mrows if r['epoch'] == mrows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last) / len(last)
    mil_first, mil_last = mrows[0]['mil'], sum(r['mil'] for r in last) / len(last)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul_last:.4f}'
    print(f'{row}: J10 {STEPS} | config diff to the Nexar parent {sorted(diff)} | mil {mil_first:.3f} -> {mil_last:.3f} | mul_mil {mul_last:.4f}')


def train_row(row):
    arm, kernel = mt.parse_row(row)
    run_dir, ckpt = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    sets = mt.train_sets(arm, kernel, SEED)
    clip_dir = L_BAKE_TRAIN if arm in mt.MOTION_ARMS else L_CLIP_TRAIN
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', clip_dir, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min ({(time.time() - t0) / STEPS:.1f} s/step)')
    check_run(row)


for row in ROWS:
    train_row(row)

## 11. MSAD test reads (`core.evaluate`, whole video, s8, raw scores, MSAD's own definitions) of the 8 rows

Motion rows (K5*) read the MSAD-statistics test bake `…/MSAD-full_test_s8_msad`; L0 rows read plain `MSAD_ncc`. Output `ROOT/eval/<row>/s2024/` (what `msad_train report` reads).

In [ ]:
L_CLIP_TEST = stage(CLIP, LOCAL / 'stage' / 'clip_test', TEST_IDS)
L_BAKE_TEST = stage(BAKE_TEST, LOCAL / 'stage' / 'bake_test', TEST_IDS)
bad = [v for v in TEST_IDS if rows(L_BAKE_TEST, v) != rows(L_CLIP_TEST, v)]
assert not bad, f'test bake rows != CLIP rows on {bad[:5]} (M3)'
assert not set(TEST_IDS) & set(TRAIN_IDS)


def is_motion(row):
    return row.startswith('K5')


def ckpt_of(row):
    return row_paths(row)[1]


def read_evaluate(row, set_name, dataset, data_dir, clip_dir, n_items, out_root):
    out = out_root / row / f's{SEED}'
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / set_name / row
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt_of(row), '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', clip_dir, '--score-norm', 'auto', '--save-scores',
             '--output-dir', local], log=LOCAL / f'eval_{set_name}_{row}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['num_videos'] == n_items and r['score_norm'] == 'none', r
    print(f"{set_name} {row}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")
    return r


for row in ROWS:
    read_evaluate(row, 'msad', DSET, L_DATA, L_BAKE_TEST if is_motion(row) else L_CLIP_TEST, len(TEST_IDS), EVAL)
flush_drive()

## 12. TAD reads (optional): macro over 60, raw scores. K5 rows read the MSAD-statistics TAD bake, L0 rows plain `TAD_ncc`

In [ ]:
if DO_TAD:
    L_TAD = LOCAL / 'TAD_meta'
    L_TAD.mkdir(parents=True, exist_ok=True)
    for f in TAD.iterdir():                     # json / txt only -- never a frame folder
        if f.is_file() and f.suffix in ('.json', '.txt'):
            shutil.copy2(f, L_TAD / f.name)
    tad_in = {True: stage(BAKE_TAD, LOCAL / 'stage' / 'tad_bake', TAD_IDS), False: stage(TAD_NCC, LOCAL / 'stage' / 'tad_ncc', TAD_IDS)}
    for row in ROWS:
        read_evaluate(row, 'tad', TAD_NAME, L_TAD, tad_in[is_motion(row)], len(TAD_IDS), EVAL_TAD)
    flush_drive()
else:
    print('DO_TAD is False -- no TAD reads')

## 13. The table -> **paste `REPORTS/k5msad_train_report.md` to Claude**

Every row of both kernels, MSAD-full / MSAD-traffic / TAD macro + micro + the no-pixel rulers, and the §41 R5 contrasts (paired per video). **No best-of, no kernel pick (R6).**

In [ ]:
local = LOCAL / 'report'
run([sys.executable, '-m', 'core.tools.msad_train', 'report', '--root', ROOT, '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_MSAD_TRAIN_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{constants.KIM_MSAD_TRAIN_REPORT_STEM}.md to Claude')